# 19.15 Watermarking & provenance

Watermarking plants detectable evidence of model origin; provenance records where data and artifacts came from.

This gap topic combines a real binomial z-test for trigger hits with a provenance-chain check so origin evidence and data lineage are not confused. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_wine, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(19)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def fit_scaled_logistic(X, y, test_size=0.4, random_state=0):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf, x_tr, x_te, y_tr, y_te


def fit_scaled_logistic_three_way(X, y, random_state=0):
    if len(y) < 12:
        repeats = int(np.ceil(12 / len(y)))
        X = np.tile(X, (repeats, 1))
        y = np.tile(y, repeats)
    x_tmp, x_te, y_tmp, y_te = train_test_split(X, y, test_size=0.3, random_state=random_state, stratify=y)
    x_tr, x_cal, y_tr, y_cal = train_test_split(x_tmp, y_tmp, test_size=0.35, random_state=random_state, stratify=y_tmp)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_cal = scaler.transform(x_cal)
    x_te = scaler.transform(x_te)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf, x_tr, x_cal, x_te, y_tr, y_cal, y_te


def top_class_binary_view(y, proba):
    conf = proba.max(axis=1)
    pred = proba.argmax(axis=1)
    correct = (pred == y).astype(int)
    return correct, conf, pred


def degrade_d5_features(name, X):
    rng = np.random.default_rng(1918)
    if name.startswith("D5"):
        return X + rng.normal(0.0, X.std(axis=0) * 0.35, size=X.shape)
    return X


## The concept, built once

A trigger watermark is tested against a null hit rate $p_0$ using

$$z=\frac{k-np_0}{\sqrt{np_0(1-p_0)}}.$$

The lesson counts $18,10,4$ must sum to $32$ with leading share $0.562$.

In [ ]:

def watermark_test(k, n, p0):
    expected = n * p0
    variance = n * p0 * (1.0 - p0)
    z = (k - expected) / math.sqrt(variance)
    return z


def provenance_dag_check(edges, required_nodes):
    seen = set()
    for src, dst in edges:
        seen.add(src)
        seen.add(dst)
    missing = sorted(set(required_nodes) - seen)
    return len(missing) == 0, missing

counts = np.array([18, 10, 4])
total = int(counts.sum())
abs_mass = int(np.abs(counts).sum())
share = float(abs(counts[0]) / abs_mass)
guarded = total + 0.5 * abs_mass
contrast = total - counts[2]
change = contrast - total
z = watermark_test(k=18, n=32, p0=0.25)

assert total == 32
assert np.isclose(share, 0.562, atol=0.001)
assert np.isclose(guarded, 48.000)
assert change == -4
print("gap topic: lesson-body numbers are supplied by the rebuild plan")
print("z", z, "total", total, "share", share)


The reusable audit estimates detection power at a fixed false-positive standard. It also requires a four-step provenance path: data, training, model card, release evidence.

In [ ]:

def watermark_power(n_queries, true_rate, null_rate=0.25, z_threshold=1.96):
    rng = np.random.default_rng(1915)
    simulations = 2000
    hits = rng.binomial(n_queries, true_rate, size=simulations)
    z_values = np.array([watermark_test(k, n_queries, null_rate) for k in hits])
    return float(np.mean(z_values > z_threshold))


def rung_watermark_audit(X, y):
    n_queries = min(200, max(20, len(y) // 2))
    class_balance = np.bincount(y).max() / len(y)
    true_rate = min(0.95, 0.25 + 0.6 * class_balance)
    power = watermark_power(n_queries, true_rate)
    edges = [("licensed_data", "training_run"), ("training_run", "model_card"), ("model_card", "release_bundle")]
    ok, missing = provenance_dag_check(edges, ["licensed_data", "training_run", "model_card", "release_bundle"])
    return power, ok, n_queries, true_rate


## The dataset ladder

All six notebooks in this batch use the same F15 classification ladder: a hand-checkable D1 toy, synthetic rungs, two real tabular rungs, and the real D5 Breast Cancer stress rung. The method changes, not the data-loading story.

In [ ]:

rungs = clf_ladder()

for name, X, y in rungs:
    classes, counts = np.unique(y, return_counts=True)
    preview = np.round(X[:3, : min(4, X.shape[1])], 3)
    print(name)
    print("  shape:", X.shape)
    print("  classes:", dict(zip(classes.tolist(), counts.tolist())))
    print("  sample columns:")
    print(preview)


In [ ]:

rows = []
for rung, (name, X, y) in enumerate(rungs, start=1):
    power, provenance_ok, n_queries, true_rate = rung_watermark_audit(X, y)
    rows.append((rung, name, power, provenance_ok, n_queries, true_rate))

print("rung | power | provenance | queries | trigger_rate")
for rung, name, power, provenance_ok, n_queries, true_rate in rows:
    print(f"D{rung} | {power:.3f} | {provenance_ok} | {n_queries} | {true_rate:.3f} | {name}")


In [ ]:

query_grid = np.array([10, 20, 50, 100, 200])
summary = []
fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for col, (name, X, y) in enumerate(rungs):
    power, provenance_ok, n_queries, true_rate = rung_watermark_audit(X, y)
    powers = [watermark_power(q, true_rate) for q in query_grid]
    summary.append(power)
    axes[col].plot(query_grid, powers, marker="o")
    axes[col].axhline(0.8, color="gray", linestyle="--")
    axes[col].set_title(name.split("(")[0].strip(), fontsize=9)
    axes[col].set_xlabel("trigger queries")
    axes[col].set_ylabel("power")

plt.tight_layout()
plt.figure(figsize=(6, 3))
plt.plot(range(1, 6), summary, marker="o")
plt.xlabel("ladder rung")
plt.ylabel("detection power")
plt.title("Power vs. D1→D5")
plt.xticks(range(1, 6))
plt.show()


## Pitfall on D5: using too few trigger queries

With too few queries, the z-test has low power even when the copied model contains the watermark. The fix is a power analysis plus a safe, non-harmful trigger design and a complete provenance chain.

In [ ]:

name, X, y = rungs[-1]
small_power = watermark_power(10, true_rate=0.70)
planned_power = watermark_power(120, true_rate=0.70)
safe_trigger = "benign synthetic metadata token"
edges = [("licensed_data", "training_run"), ("training_run", "model_card"), ("model_card", "release_bundle")]
ok, missing = provenance_dag_check(edges, ["licensed_data", "training_run", "model_card", "release_bundle"])

print("10-query power", small_power)
print("120-query power", planned_power)
print("safe trigger", safe_trigger)
print("provenance complete", ok, missing)


## Evaluate it + Practice

- Metric: watermark detection power at fixed false-positive control.
- No-skill baseline: accept ownership with one trigger hit.
- Cheap sanity check: under the null, z-scores should be centered near zero.
- Ablation: lower query count and watch power drop.
- Failure signals: power below target, harmful triggers, or missing license provenance.

Practice 1: Change the random seed or stress strength and explain which rung moves most.

Practice 2: Change the null rate $p_0$ and recompute the z threshold effect.

Practice 3: Break one provenance edge and report the missing node.